# Evaluación de Precisión del Sistema

**Objetivo:** Evaluar la precisión de los resultados generados por el analizador de complejidades
**Duración estimada:** 40 minutos

---

## Contenido

1. [Setup](#setup)
2. [Casos de Referencia con Resultados Conocidos](#casos-de-referencia)
3. [Evaluación del Analizador de Complejidad](#evaluacion-del-analizador)
4. [Evaluación del Detector de Patrones](#evaluacion-del-detector-de-patrones)
5. [Evaluación del Detector de Estructuras](#evaluacion-del-detector-de-estructuras)
6. [Matriz de Confusión de Patrones](#matriz-de-confusion)
7. [Métricas Globales de Precisión](#metricas-globales)
8. [Casos Problemáticos](#casos-problematicos)

---

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '../..')

from app.core.parser import parse_pseudocode
from app.core.analyzer import AnalyzerEngine
from app.core.patterns import PatternDetector, PatternType
from app.core.data_structures import StructureIdentifier, StructureType

import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

print("Setup completado")

---

## 2. Casos de Referencia con Resultados Conocidos

Los casos de referencia son algoritmos clásicos cuya complejidad, patrones y estructuras de datos son bien conocidos.
Sirven como ground truth para evaluar la precisión del sistema.

In [ ]:
# Formato: (codigo, big_o_esperado, patron_esperado, estructuras_esperadas)
CASOS_REFERENCIA = [
    {
        "nombre": "Búsqueda Lineal",
        "codigo": """
algorithm linearSearch(A[], n, key)
begin
    for i <- 1 to n do
        if (A[i] = key) then
            return i
        end
    end
    return -1
end
""",
        "big_o_esperado": "O(n)",
        "omega_esperado": "O(1)",
        "patron_esperado": PatternType.SEARCHING,
        "estructuras_esperadas": ["ARRAY"]
    },
    {
        "nombre": "Búsqueda Binaria",
        "codigo": """
algorithm binarySearch(A[], low, high, key)
begin
    while (low <= high) do
        mid <- floor((low + high) / 2)
        if (A[mid] = key) then
            return mid
        end
        if (A[mid] < key) then
            low <- mid + 1
        else
            high <- mid - 1
        end
    end
    return -1
end
""",
        "big_o_esperado": "O(log n)",
        "omega_esperado": "O(1)",
        "patron_esperado": PatternType.SEARCHING,
        "estructuras_esperadas": ["ARRAY"]
    },
    {
        "nombre": "Bubble Sort",
        "codigo": """
algorithm bubbleSort(A[], n)
begin
    for i <- 1 to n - 1 do
        for j <- 1 to n - i do
            if (A[j] > A[j + 1]) then
                temp <- A[j]
                A[j] <- A[j + 1]
                A[j + 1] <- temp
            end
        end
    end
end
""",
        "big_o_esperado": "O(n^2)",
        "omega_esperado": "O(n)",
        "patron_esperado": PatternType.SORTING,
        "estructuras_esperadas": ["ARRAY"]
    },
    {
        "nombre": "Fibonacci Recursivo",
        "codigo": """
algorithm fibonacci(n)
begin
    if (n <= 1) then
        return n
    end
    return fibonacci(n - 1) + fibonacci(n - 2)
end
""",
        "big_o_esperado": "O(2^n)",
        "omega_esperado": "O(1)",
        "patron_esperado": PatternType.RECURSIVE,
        "estructuras_esperadas": []
    },
    {
        "nombre": "Merge Sort",
        "codigo": """
algorithm mergeSort(A[], p, r)
begin
    if (p < r) then
        q <- floor((p + r) / 2)
        call mergeSort(A, p, q)
        call mergeSort(A, q + 1, r)
    end
end
""",
        "big_o_esperado": "O(n log n)",
        "omega_esperado": "O(n log n)",
        "patron_esperado": PatternType.DIVIDE_AND_CONQUER,
        "estructuras_esperadas": ["ARRAY"]
    },
]

print(f"Casos de referencia cargados: {len(CASOS_REFERENCIA)}")
for caso in CASOS_REFERENCIA:
    print(f"  - {caso['nombre']}: BigO={caso['big_o_esperado']}, Patron={caso['patron_esperado']}")

---

## 3. Evaluación del Analizador de Complejidad

In [ ]:
def normalizar_big_o(notacion):
    """Normaliza la notación Big O para comparación."""
    mapa = {
        "o(n)": "O(n)",
        "o(n^2)": "O(n^2)",
        "o(n²)": "O(n^2)",
        "o(2^n)": "O(2^n)",
        "o(log n)": "O(log n)",
        "o(n log n)": "O(n log n)",
        "o(n*log(n))": "O(n log n)",
        "o(1)": "O(1)",
    }
    if notacion is None:
        return None
    return mapa.get(str(notacion).lower().replace(" ", ""), str(notacion))

def evaluar_complejidad(casos):
    """Evalúa la precisión del analizador de complejidad temporal."""
    engine = AnalyzerEngine()
    resultados = []
    
    for caso in casos:
        try:
            ast = parse_pseudocode(caso["codigo"])
            analysis = engine.analyze(ast)
            
            big_o_obtenido = normalizar_big_o(
                getattr(analysis, 'big_o', None) or
                getattr(analysis, 'time_complexity', {}).get('big_o')
            )
            big_o_esperado = caso["big_o_esperado"]
            
            correcto = big_o_obtenido == big_o_esperado
            
            resultados.append({
                "nombre": caso["nombre"],
                "esperado": big_o_esperado,
                "obtenido": big_o_obtenido,
                "correcto": correcto
            })
        except Exception as e:
            resultados.append({
                "nombre": caso["nombre"],
                "esperado": caso["big_o_esperado"],
                "obtenido": f"ERROR: {str(e)[:40]}",
                "correcto": False
            })
    
    return resultados

print("Evaluando precisión del analizador de complejidad...")
resultados_complejidad = evaluar_complejidad(CASOS_REFERENCIA)

print("\nRESULTADOS ANALIZADOR DE COMPLEJIDAD:")
print(f"{'Algoritmo':<25} {'Esperado':<15} {'Obtenido':<15} {'Estado':>8}")
correctos = 0
for r in resultados_complejidad:
    estado = "PASS" if r["correcto"] else "FAIL"
    if r["correcto"]:
        correctos += 1
    print(f"{r['nombre']:<25} {r['esperado']:<15} {str(r['obtenido']):<15} {estado:>8}")

precision = correctos / len(resultados_complejidad) * 100
print(f"\nPrecisión total: {correctos}/{len(resultados_complejidad)} = {precision:.1f}%")

---

## 4. Evaluación del Detector de Patrones

In [ ]:
def evaluar_patrones(casos):
    """Evalúa la precisión del detector de patrones."""
    detector = PatternDetector()
    resultados = []
    
    for caso in casos:
        try:
            ast = parse_pseudocode(caso["codigo"])
            detection = detector.detect(ast)
            
            patron_obtenido = getattr(detection, 'primary_pattern', None)
            patron_esperado = caso["patron_esperado"]
            
            correcto = patron_obtenido == patron_esperado
            confianza = getattr(detection, 'primary_confidence', 0.0)
            
            resultados.append({
                "nombre": caso["nombre"],
                "esperado": patron_esperado,
                "obtenido": patron_obtenido,
                "confianza": confianza,
                "correcto": correcto
            })
        except Exception as e:
            resultados.append({
                "nombre": caso["nombre"],
                "esperado": caso["patron_esperado"],
                "obtenido": None,
                "confianza": 0.0,
                "correcto": False
            })
    
    return resultados

print("Evaluando precisión del detector de patrones...")
resultados_patrones = evaluar_patrones(CASOS_REFERENCIA)

print("\nRESULTADOS DETECTOR DE PATRONES:")
print(f"{'Algoritmo':<25} {'Esperado':<20} {'Confianza':>10} {'Estado':>8}")
correctos_p = 0
for r in resultados_patrones:
    estado = "PASS" if r["correcto"] else "FAIL"
    if r["correcto"]:
        correctos_p += 1
    patron_obtenido_str = str(r['obtenido'])[:18] if r['obtenido'] else "None"
    print(f"{r['nombre']:<25} {patron_obtenido_str:<20} {r['confianza']:>9.2f} {estado:>8}")

precision_p = correctos_p / len(resultados_patrones) * 100
print(f"\nPrecisión total: {correctos_p}/{len(resultados_patrones)} = {precision_p:.1f}%")

---

## 5. Evaluación del Detector de Estructuras

In [ ]:
def evaluar_estructuras(casos):
    """Evalúa la precisión del detector de estructuras de datos."""
    identificador = StructureIdentifier()
    resultados = []
    
    for caso in casos:
        try:
            ast = parse_pseudocode(caso["codigo"])
            detection = identificador.identify(ast)
            
            estructuras_obtenidas = set(
                str(m.structure_type).replace("StructureType.", "")
                for m in getattr(detection, 'structures', [])
            )
            estructuras_esperadas = set(caso["estructuras_esperadas"])
            
            precision_local = (
                len(estructuras_obtenidas & estructuras_esperadas) /
                max(len(estructuras_esperadas), 1)
                if estructuras_esperadas else 1.0
            )
            
            resultados.append({
                "nombre": caso["nombre"],
                "esperadas": estructuras_esperadas,
                "obtenidas": estructuras_obtenidas,
                "precision": precision_local,
                "correcto": precision_local == 1.0
            })
        except Exception as e:
            resultados.append({
                "nombre": caso["nombre"],
                "esperadas": set(caso["estructuras_esperadas"]),
                "obtenidas": set(),
                "precision": 0.0,
                "correcto": False
            })
    
    return resultados

print("Evaluando precisión del detector de estructuras...")
resultados_estructuras = evaluar_estructuras(CASOS_REFERENCIA)

print("\nRESULTADOS DETECTOR DE ESTRUCTURAS:")
for r in resultados_estructuras:
    estado = "PASS" if r["correcto"] else f"PARCIAL({r['precision']:.0%})"
    print(f"  {r['nombre']}: esperadas={r['esperadas']}, obtenidas={r['obtenidas']} -> {estado}")

---

## 6. Matriz de Confusión de Patrones

In [ ]:
# Construir matriz de confusión para detección de patrones
from collections import Counter

patrones_unicos = list(set([c["patron_esperado"] for c in CASOS_REFERENCIA]))
n = len(patrones_unicos)
patron_a_idx = {p: i for i, p in enumerate(patrones_unicos)}

matriz = np.zeros((n, n), dtype=int)
for r in resultados_patrones:
    idx_esperado = patron_a_idx.get(r["esperado"])
    idx_obtenido = patron_a_idx.get(r["obtenido"])
    if idx_esperado is not None and idx_obtenido is not None:
        matriz[idx_esperado][idx_obtenido] += 1

# Visualizar
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(matriz, cmap="Blues")

etiquetas = [str(p).replace("PatternType.", "")[:12] for p in patrones_unicos]
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(etiquetas, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(etiquetas, fontsize=9)
ax.set_xlabel("Patrón Detectado")
ax.set_ylabel("Patrón Real")
ax.set_title("Matriz de Confusión - Detección de Patrones")

for i in range(n):
    for j in range(n):
        ax.text(j, i, str(matriz[i, j]), ha="center", va="center",
                color="white" if matriz[i, j] > 0.5 else "black")

plt.colorbar(im)
plt.tight_layout()
plt.savefig("confusion_matrix_patrones.png", dpi=120, bbox_inches="tight")
plt.show()

---

## 7. Métricas Globales de Precisión

In [ ]:
print("RESUMEN DE PRECISIÓN DEL SISTEMA")

prec_complejidad = sum(1 for r in resultados_complejidad if r["correcto"]) / len(resultados_complejidad)
prec_patrones = sum(1 for r in resultados_patrones if r["correcto"]) / len(resultados_patrones)
prec_estructuras = sum(r["precision"] for r in resultados_estructuras) / len(resultados_estructuras)

print(f"\nAnalizador de Complejidad (Big O): {prec_complejidad:.1%}")
print(f"Detector de Patrones:              {prec_patrones:.1%}")
print(f"Detector de Estructuras:           {prec_estructuras:.1%}")
print(f"\nPrecisión Global (promedio):       {(prec_complejidad + prec_patrones + prec_estructuras) / 3:.1%}")

# Confianza promedio del detector de patrones
conf_promedio = sum(r["confianza"] for r in resultados_patrones) / len(resultados_patrones)
print(f"Confianza promedio en detección:   {conf_promedio:.2f}")

---

## 8. Casos Problemáticos

In [ ]:
print("\nCASOS QUE REQUIEREN REVISIÓN:")

problemas_encontrados = False

for r in resultados_complejidad:
    if not r["correcto"]:
        print(f"[Complejidad] {r['nombre']}: esperado={r['esperado']}, obtenido={r['obtenido']}")
        problemas_encontrados = True

for r in resultados_patrones:
    if not r["correcto"]:
        print(f"[Patron] {r['nombre']}: esperado={r['esperado']}, obtenido={r['obtenido']}")
        problemas_encontrados = True

for r in resultados_estructuras:
    if not r["correcto"]:
        faltantes = r["esperadas"] - r["obtenidas"]
        extras = r["obtenidas"] - r["esperadas"]
        print(f"[Estructura] {r['nombre']}: faltantes={faltantes}, extras={extras}")
        problemas_encontrados = True

if not problemas_encontrados:
    print("No se encontraron problemas en los casos de referencia.")
    print("Considerar ampliar el conjunto de casos para mayor cobertura.")

---

## Proximos Pasos

- **error_analysis.ipynb**: Análisis detallado de los casos de error detectados
- **performance_metrics.ipynb**: Medir el rendimiento del sistema